# Semantic Router Evaluation Pipeline

This notebook evaluates the `SemanticRouterModule` performance. It classifies legal queries into **Reasoning-LLM**, **General-LLM**, or **Casual-LLM** routes.

**Current Task:** Benchmarking between Qwen-2.5 and Gemini-2.5 Flash Lite as requested by the team leader.

In [6]:
# 1. Install Dependencies
%pip install pandas tqdm requests python-dotenv rich scikit-learn -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# ==================================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==================================================================
import sys, os, time, json, re, types
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from sklearn.metrics import classification_report

# --- Path Management ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.append(project_root)
load_dotenv(os.path.join(project_root, '.env'))

# --- Framework Imports ---
from src.adaptive_routing.modules.triage import TriageModule
from src.adaptive_routing.modules.router import SemanticRouterModule
from src.adaptive_routing.config import FrameworkConfig

# --- Settings ---
OPENROUTER_API_KEY = ""  # Leave empty to use .env file
QUERY_LIMIT = None  # Set to None for full evaluation
SELECTED_MODEL = "Qwen-2.5" # @param ["Qwen-2.5", "Gemini-2.5"]

MODEL_OPTIONS = {
    "Qwen-2.5": "qwen/qwen-2.5-7b-instruct",
    "Gemini-2.5": "google/gemini-2.0-flash-lite-001"
}

CONFIG = {
    "api_key": OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY"),
    "triage_model": MODEL_OPTIONS[SELECTED_MODEL], 
    "router_model": MODEL_OPTIONS[SELECTED_MODEL],
    "router_temp": 0.1,                   
    "router_max_tokens": 250,         
    "router_use_system": True
}

SYSTEM_PROMPT = """ROLE: Legal Query Router
TASK: Analyze the USER QUERY and decide which LLM should handle it.

STRICT CONSTRAINTS:
- Return ONLY a valid JSON object. 
- Do NOT include any conversational text, explanations, or opening/closing remarks.
- Do NOT answer the user's legal question. Your only job is to route it.
- Strictly adhere to the JSON Schema below.

JSON Schema:
{
  \"route\": \"Casual-LLM\" | \"General-LLM\" | \"Reasoning-LLM\",
  \"confidence\": float,
  \"search_signals\": [list of short phrases] | null
}"""

# --- Initialization ---
FrameworkConfig._update_settings_(**CONFIG)
triage = TriageModule(api_key=CONFIG['api_key'])
router = SemanticRouterModule(api_key=CONFIG['api_key'])

# --- ROBUSTNESS PATCH: Injected JSON Parser ---
def _robust_parse_json(self, text: str) -> dict:
    if not isinstance(text, str): text = str(text) if text is not None else ""
    # 1. Try to extract content between ```json and ```
    json_match = re.search(r'```json\\s*({.*?})\\s*```', text, re.DOTALL | re.IGNORECASE)
    if not json_match:
        # 2. Try to extract content between first { and last }
        json_match = re.search(r'({.*})', text, re.DOTALL)
    
    if json_match:
        try: return json.loads(json_match.group(1))
        except: pass
            
    # 3. Fallback: Standard stripping
    cleaned = re.sub(r"```json\\s*", "", text, flags=re.IGNORECASE)
    cleaned = re.sub(r"```", "", cleaned).strip()
    try: return json.loads(cleaned)
    except:
        return {"route": None, "confidence": 0.0, "error": "Injected parser failed to extract JSON"}

router._classifier._parse_json_ = types.MethodType(_robust_parse_json, router._classifier)
router._classifier._system_prompt = SYSTEM_PROMPT

print(f"Engine ready with model: {FrameworkConfig._ROUTER_MODEL}")
print("Robust JSON parser and Strict System Prompt injected.")

Engine ready with model: qwen/qwen-2.5-7b-instruct
Robust JSON parser and Strict System Prompt injected.


In [8]:
# ==================================================================
# 2. LOAD DATASET & MANAGE CHECKPOINTS
# ==================================================================
dataset_path = 'dataset/Routing-Evaluation-Dataset.csv'
input_column = 'Query'
label_column = 'Groq Expected'
pred_col = f"Predicted_{SELECTED_MODEL}"
checkpoint_path = f'dataset/Routing-Evaluation-Checkpoint-{SELECTED_MODEL}.csv'

# Load or Resume from Checkpoint
if os.path.exists(checkpoint_path):
    print(f"Resuming from checkpoint: {checkpoint_path}")
    df = pd.read_csv(checkpoint_path)
    if pred_col not in df.columns:
        df[pred_col] = None
else:
    print(f"Loading fresh dataset: {dataset_path}")
    df = pd.read_csv(dataset_path)
    df[pred_col] = None

# Apply Query Limit if set in configurations
if QUERY_LIMIT:
    df = df.head(QUERY_LIMIT)

print(f"Dataset Ready: {len(df)} rows loaded.")

Resuming from checkpoint: dataset/Routing-Evaluation-Checkpoint-Qwen-2.5.csv
Dataset Ready: 581 rows loaded.


In [4]:
# ==================================================================
# 3. QUICK TEST (Single Inference)
# ==================================================================
sample_query = "How do I file a case for illegal dismissal?"

print(f"Testing: '{sample_query}'")
try:
    start_time = time.time()
    triage_res = triage._process_request_(sample_query)
    norm_text = triage_res.get("normalized_text", sample_query)
    
    route_res = router._process_routing_(norm_text)
    duration = time.time() - start_time
    
    print("\n--- RESULTS ---")
    print(f"Detected Route: {route_res.get('route')}")
    print(f"Confidence: {route_res.get('confidence')}")
    print(f"Latency: {duration:.2f} seconds")
except Exception as e:
    print(f"Error: {e}")

Testing: 'How do I file a case for illegal dismissal?'


Language tag not found in normalizer output. First 100 chars: Filing a case for illegal dismissal can vary depending on the country and the specific labor laws. H



--- RESULTS ---
Detected Route: General-LLM
Confidence: 0.85
Latency: 17.24 seconds


In [9]:
# ==================================================================
# 4. EXECUTE EVALUATION (With Retry Logic & Increment Logic)
# ==================================================================
pred_col = f"Predicted_{SELECTED_MODEL}"
checkpoint_path = f'dataset/Routing-Evaluation-Checkpoint-{SELECTED_MODEL}.csv'

print(f"Starting evaluation for {SELECTED_MODEL}...")
for index, row in tqdm(df.iterrows(), total=len(df)):
    current_val = row.get(pred_col)
    
    # --- INCREMENTAL LOGIC ---
    # Skip only if row has a valid non-error prediction
    if pd.notna(current_val) and str(current_val).strip() != "" and not str(current_val).startswith("ERROR"):
        continue
    
    # --- RETRY LOGIC (3 attempts) ---
    for attempt in range(1, 4):
        try:
            query = str(row[input_column])
            t_res = triage._process_request_(query)
            norm_text = t_res.get("normalized_text", query)
            r_res = router._process_routing_(norm_text)
            
            if r_res.get("route"):
                df.at[index, pred_col] = r_res.get('route')
                break # Success
            else:
                raise ValueError("Empty route returned")
                
        except Exception as e:
            if attempt == 3:
                df.at[index, pred_col] = f"ERROR: {str(e)[:50]}"
            else:
                time.sleep(2 ** attempt) # Exponential backoff
    
    if (index + 1) % 10 == 0: df.to_csv(checkpoint_path, index=False)

print("Evaluation cycle complete.")

Starting evaluation for Qwen-2.5...


  5%|▌         | 31/581 [00:17<05:10,  1.77it/s]Language tag not found in normalizer output. First 100 chars: The contract was signed before September 30, 2025. Will I be entitled to the $5,100 salary?

Detecte
  6%|▌         | 32/581 [00:21<07:16,  1.26it/s]Language tag not found in normalizer output. First 100 chars: Sa Pilipinas, ang mga statutory holidays ay maaaring ipunin o mabawasan kung hindi ito ang mga araw 
Language tag not found in normalizer output. First 100 chars: Oo, kaya mo itipunin ang mga statutory holidays sa dulo ng taon. Ito ay maaaring makatulong sa iyo u
 19%|█▉        | 112/581 [03:05<24:14,  3.10s/it]Language tag not found in normalizer output. First 100 chars: You should consult with HK authorities regarding your complaint about illegal collection. 

Detected
 25%|██▌       | 146/581 [04:06<13:15,  1.83s/it]Language tag not found in normalizer output. First 100 chars: How can one report an illegal recruitment agency back in the Philippines?

Detected Raw Lang

Evaluation cycle complete.


In [10]:
# 6. Summary & Results
output_path = f'dataset/Routing-Evaluation-Results-{SELECTED_MODEL}.csv'
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

if 'Groq Expected' in df.columns:
    eval_df = df[df[pred_col].notna() & (~df[pred_col].str.startswith("ERROR"))]
    if not eval_df.empty:
        accuracy = (eval_df[pred_col] == eval_df['Groq Expected']).mean() * 100
        print(f"\n{SELECTED_MODEL} Accuracy: {accuracy:.2f}%")
        
        print("\nClassification Report:")
        report = classification_report(eval_df['Groq Expected'], eval_df[pred_col])
        print(report)
        
        print("\nDistribution:")
        print(eval_df[pred_col].value_counts())
    else:
        print("No valid predictions to evaluate.")
df.head()

Results saved to dataset/Routing-Evaluation-Results-Qwen-2.5.csv

Qwen-2.5 Accuracy: 65.40%

Classification Report:
               precision    recall  f1-score   support

   Casual-LLM       0.00      0.00      0.00         0
  General-LLM       0.62      0.78      0.69       283
Reasoning-LLM       0.72      0.53      0.61       298

     accuracy                           0.65       581
    macro avg       0.45      0.44      0.43       581
 weighted avg       0.67      0.65      0.65       581


Distribution:
Predicted_Qwen-2.5
General-LLM      357
Reasoning-LLM    221
Casual-LLM         3
Name: count, dtype: int64


c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

,Query,Expected,Groq Expected,Groq Reason,Predicted,Predicted_Qwen-2.5
0,Tatlong OFW na nasa deathrow sa China... Naha...,Reasoning-LLM,Reasoning-LLM,The query involves a specific scenario of OFWs...,NaN,Reasoning-LLM
1,Ang Passport and Employment Contract ko ay nas...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation with p...,NaN,Reasoning-LLM
2,pag may pandemic. maiiwan ba ako here sa hongk...,General-LLM,Reasoning-LLM,The query involves a personal situation of an ...,NaN,General-LLM
3,Paano gumagana yung bagong Senate Bill No. 2390?,General-LLM,General-LLM,The query asks for general information about t...,NaN,General-LLM
4,Nawalan ako ng trabaho dahil I made a Case sa ...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation and as...,NaN,Reasoning-LLM
